In [ ]:
%%sql -r dataframe_2
select * from  snowflake_sample_data.tpch_sf1.customer

In [ ]:
%%sql -r dataframe_1
--Check the range of values in the Market Segment Column
select distinct c_mktsegment
from snowflake_sample_data.tpch_sf1.customer;

--Find out which Market Segments have the most customers
select c_mktsegment, count(*)
from snowflake_sample_data.tpch_sf1.customer
group by c_mktsegment
order by count(*);

In [ ]:
%%sql -r dataframe_3
-- Nations Table
select n_nationkey, n_name, n_regionkey
from snowflake_sample_data.tpch_sf1.nation;

In [ ]:
%%sql -r dataframe_4
-- Regions Table
select r_regionkey, r_name
from snowflake_sample_data.tpch_sf1.region;

In [ ]:
%%sql -r dataframe_5
-- Join the Tables and Sort
select r_name as region, n_name as nation
from snowflake_sample_data.tpch_sf1.nation
join snowflake_sample_data.tpch_sf1.region
on n_regionkey = r_regionkey
order by r_name, n_name asc;

In [ ]:
%%sql -r dataframe_6
select r_name as region, count(n_name) as num_countries
from snowflake_sample_data.tpch_sf1.nation
join snowflake_sample_data.tpch_sf1.region
on n_regionkey = r_regionkey
group by r_name;

In [ ]:
%%sql -r dataframe_7
use role SYSADMIN;

create database INTL_DB;

use schema INTL_DB.PUBLIC;

In [ ]:
%%sql -r dataframe_8
use role SYSADMIN;

create warehouse INTL_WH 
with 
warehouse_size = 'XSMALL' 
warehouse_type = 'STANDARD' 
auto_suspend = 600 --600 seconds/10 mins
auto_resume = TRUE;

use warehouse INTL_WH;

In [ ]:
%%sql -r dataframe_9
create or replace table intl_db.public.INT_STDS_ORG_3166 
(iso_country_name varchar(100), 
 country_name_official varchar(200), 
 sovreignty varchar(40), 
 alpha_code_2digit varchar(2), 
 alpha_code_3digit varchar(3), 
 numeric_country_code integer,
 iso_subdivision varchar(15), 
 internet_domain_code varchar(10)
);

In [ ]:
%%sql -r dataframe_10
create or replace file format util_db.public.PIPE_DBLQUOTE_HEADER_CR 
  type = 'CSV' 
  compression = 'AUTO' 
  field_delimiter = '|' --pipe 
  record_delimiter = '\r' --carriage return
  skip_header = 1  --1 header row
  field_optionally_enclosed_by = '\042'
  trim_space = FALSE;

In [ ]:
%%sql -r dataframe_11
show stages in account;

In [ ]:
%%sql -r dataframe_12
list @util_db.public.MY_INTERNAL_STGAE

In [ ]:
%%sql -r dataframe_13
create table intl_db.public.CURRENCIES 
(
  currency_ID integer, 
  currency_char_code varchar(3), 
  currency_symbol varchar(4), 
  currency_digital_code varchar(3), 
  currency_digital_name varchar(30)
)
  comment = 'Information about currencies including character codes, symbols, digital codes, etc.';

In [ ]:
%%sql -r dataframe_14
create table intl_db.public.COUNTRY_CODE_TO_CURRENCY_CODE 
  (
    country_char_code varchar(3), 
    country_numeric_code integer, 
    country_name varchar(100), 
    currency_name varchar(100), 
    currency_char_code varchar(3), 
    currency_numeric_code integer
  ) 
  comment = 'Mapping table currencies to countries';

In [ ]:
%%sql -r dataframe_15
 create file format util_db.public.CSV_COMMA_LF_HEADER
  type = 'CSV' 
  field_delimiter = ',' 
  record_delimiter = '\n' -- the n represents a Line Feed character
  skip_header = 1 
;

In [ ]:
%%sql -r dataframe_16
copy into intl_db.public.CURRENCIES 
from @util_db.public.MY_INTERNAL_STGAE
files = ( 'country_code_to_currency_code.xls')
file_format = ( format_name='util_db.public.CSV_COMMA_LF_HEADER' );

In [ ]:
copy into intl_db.public.COUNTRY_CODE_TO_CURRENCY_CODE 
from @util_db.public.MY_INTERNAL_STGAE
files = ( 'country_code_to_currency_code.xls')
file_format = ( format_name='util_db.public.CSV_COMMA_LF_HEADER' );

In [ ]:
select * from intl_db.public.COUNTRY_CODE_TO_CURRENCY_CODE 

In [ ]:
create view intl_db.public.simple_currency AS
select COUNTRY_CHAR_CODE as CTY_CODE, CURRENCY_CHAR_CODE as CUR_CODE from intl_db.public.COUNTRY_CODE_TO_CURRENCY_CODE 
;

In [ ]:
%%sql -r dataframe_21
select * from intl_db.public.simple_currency

In [ ]:
select * from intl_db.public.CURRENCIES 